In [1]:
import numpy as np
import pandas as pd
import torch
import timesfm
import logging
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from scipy.stats import pearsonr

logging.basicConfig(
    filename='TimesFM_SSMI_FineTuned_Decomposed_Kalman_v2_Metrics.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    force=True
)

CHECKPOINT_LOW = Path('./timesfm_ssmi_low_kalman_v2.pt')
CHECKPOINT_HIGH = Path('./timesfm_ssmi_high_kalman_v2.pt')

# Must match training config (finetune_timesfm_decomposed_kalman_v2.py)
KALMAN_PROCESS_NOISE = 0.1
KALMAN_MEASUREMENT_NOISE = 1.0


def kalman_decompose_context(y_context, process_noise=KALMAN_PROCESS_NOISE, measurement_noise=KALMAN_MEASUREMENT_NOISE):
    """One-pass causal scalar Kalman filter \u2014 identical to training-time decomposition."""
    y = np.asarray(y_context, dtype=float)
    q = float(process_noise)
    r = float(measurement_noise)

    x = float(y[0])
    p = 1.0
    low = np.zeros_like(y, dtype=float)

    for i, z in enumerate(y):
        p = p + q
        k = p / (p + r)
        x = x + k * (float(z) - x)
        p = (1.0 - k) * p
        low[i] = x

    high = y - low
    return low, high


def main():
    rmse_list = []
    mape_list = []
    pearson_list = []
    directional_hits = []
    try:
        # ========================
        # 1) Load SSMI test slice (post-2018)
        # ========================
        df = pd.read_csv('../../DataSets/trimmed/SSMI.csv', parse_dates=['Date'])
        df = df.sort_values('Date').reset_index(drop=True)
        test_df = df[df['Date'] >= pd.Timestamp('2018-01-01')].reset_index(drop=True)

        nan_count = int(test_df['Adj Close'].isna().sum())
        if nan_count > 0:
            test_df = test_df.copy()
            test_df['Adj Close'] = test_df['Adj Close'].ffill().bfill()
            logging.info(f'Forward-filled {nan_count} NaN values in test data')
            print(f'Forward-filled {nan_count} NaN values in test data')

        y = test_df['Adj Close'].values.astype(float)
        total_samples = len(y)
        logging.info(
            f"Test range: {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()} ({total_samples} days)"
        )
        print(
            f"Test range: {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()} ({total_samples} days)"
        )

        # ========================
        # 2) Sliding-window config
        # ========================
        context_window = 120
        forecast_horizon = 30
        step_size = 30
        num_segments = (total_samples - context_window) // step_size
        logging.info(f'Segments to evaluate: {num_segments}')

        # ========================
        # 3) Load two fine-tuned TimesFM models (low / high)
        # ========================
        tfm_low = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend='cpu',
                per_core_batch_size=32,
                horizon_len=forecast_horizon,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id='google/timesfm-1.0-200m-pytorch',
            ),
        )
        tfm_low._model.load_state_dict(torch.load(str(CHECKPOINT_LOW), map_location='cpu'))
        tfm_low._model.eval()
        logging.info(f'Fine-tuned low-pass weights loaded from {CHECKPOINT_LOW}')
        print(f'Fine-tuned low-pass weights loaded from {CHECKPOINT_LOW}')

        tfm_high = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend='cpu',
                per_core_batch_size=32,
                horizon_len=forecast_horizon,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id='google/timesfm-1.0-200m-pytorch',
            ),
        )
        tfm_high._model.load_state_dict(torch.load(str(CHECKPOINT_HIGH), map_location='cpu'))
        tfm_high._model.eval()
        logging.info(f'Fine-tuned high-pass weights loaded from {CHECKPOINT_HIGH}')
        print(f'Fine-tuned high-pass weights loaded from {CHECKPOINT_HIGH}')

        # ========================
        # 4) Sliding-window evaluation with per-window normalization
        # ========================
        for segment in range(num_segments):
            start_context = segment * step_size
            end_context = start_context + context_window
            if end_context + forecast_horizon > total_samples:
                break

            y_context = y[start_context:end_context]
            y_true = y[end_context:end_context + forecast_horizon]

            # Per-window stats from the RAW context (matches training).
            ctx_mean = float(y_context.mean())
            ctx_std = float(y_context.std())
            if ctx_std < 1e-6:
                ctx_std = 1.0

            # Kalman-decompose the RAW context, then normalize each component
            # with the RAW context stats so that low_norm + high_norm == (raw - mean)/std.
            context_low_raw, context_high_raw = kalman_decompose_context(y_context)
            context_low_norm = (context_low_raw - ctx_mean) / ctx_std
            context_high_norm = (context_high_raw - ctx_mean) / ctx_std

            point_forecast_low, _ = tfm_low.forecast([context_low_norm], freq=[0])
            forecast_low_norm = point_forecast_low[0][:forecast_horizon]

            point_forecast_high, _ = tfm_high.forecast([context_high_norm], freq=[0])
            forecast_high_norm = point_forecast_high[0][:forecast_horizon]

            forecast_low = forecast_low_norm * ctx_std + ctx_mean
            forecast_high = forecast_high_norm * ctx_std + ctx_mean
            combined_pred = forecast_low + forecast_high

            prev_anchor = np.concatenate([[y[end_context - 1]], y_true[:-1]])
            actual_direction = np.sign(y_true - prev_anchor)
            pred_direction = np.sign(combined_pred - prev_anchor)
            hits = (actual_direction == pred_direction).astype(int)
            directional_hits.extend(hits.tolist())

            rmse = np.sqrt(mean_squared_error(y_true, combined_pred))
            mape = mean_absolute_percentage_error(y_true, combined_pred) * 100
            r2 = pearsonr(y_true, combined_pred).statistic ** 2

            rmse_list.append(rmse)
            mape_list.append(mape)
            pearson_list.append(r2)

            segment_dir_acc = hits.mean() * 100
            logging.info(
                f'Segment {segment+1}/{num_segments}: RMSE={rmse:.4f}, MAPE={mape:.4f}%, R2={r2:.4f}, DirAcc={segment_dir_acc:.1f}%'
            )
            print(
                f'Segment {segment+1}/{num_segments} - RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | R2: {r2:.4f} | Dir Acc: {segment_dir_acc:.1f}%'
            )

        np.savez_compressed(
            'TimesFM_SSMI_FineTuned_Decomposed_Kalman_v2_Metrics.npz',
            rmse=np.array(rmse_list),
            mape=np.array(mape_list),
            pearson_coefficients=np.array(pearson_list),
            directional_hits=np.array(directional_hits),
            context_window=context_window,
            forecast_horizon=forecast_horizon,
            process_noise=KALMAN_PROCESS_NOISE,
            measurement_noise=KALMAN_MEASUREMENT_NOISE,
            num_segments=num_segments,
        )
        logging.info('Results saved to TimesFM_SSMI_FineTuned_Decomposed_Kalman_v2_Metrics.npz')

        total_days = len(directional_hits)
        total_hits = sum(directional_hits)
        dir_acc_pct = (total_hits / total_days) * 100 if total_days else 0.0

        print('\n--- Median Metrics for Kalman-v2 Fine-Tuned TimesFM on SSMI (post-2018 test) ---')
        print(f'Median RMSE:          {np.median(rmse_list):.4f}')
        print(f'Median MAPE:          {np.median(mape_list):.4f}%')
        print(f'Median Pearson R2:    {np.median(pearson_list):.4f}')
        print(f'Directional Accuracy: {total_hits}/{total_days} days ({dir_acc_pct:.2f}%)')

    except Exception:
        logging.error('An error occurred.', exc_info=True)
        print('An error occurred. Check TimesFM_SSMI_FineTuned_Decomposed_Kalman_v2_Metrics.log for details.')
        try:
            np.savez_compressed(
                'partial_TimesFM_SSMI_FineTuned_Decomposed_Kalman_v2_Metrics.npz',
                rmse=np.array(rmse_list),
                mape=np.array(mape_list),
                pearson_coefficients=np.array(pearson_list),
                directional_hits=np.array(directional_hits),
            )
        except Exception:
            logging.error('Failed to save partial results.', exc_info=True)
    finally:
        logging.info('Forecasting run completed.')


if __name__ == '__main__':
    main()

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


Loaded PyTorch TimesFM, likely because python version is 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].


Forward-filled 7 NaN values in test data
Test range: 2018-01-03 -> 2021-05-17 (843 days)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fine-tuned low-pass weights loaded from timesfm_ssmi_low_kalman_v2.pt


...

Segment 20/24 - RMSE: 342.71 | MAPE: 2.45% | R2: 0.2939 | Dir Acc: 60.0%


Segment 21/24 - RMSE: 121.65 | MAPE: 0.78% | R2: 0.0396 | Dir Acc: 60.0%


Segment 22/24 - RMSE: 334.43 | MAPE: 2.91% | R2: 0.1774 | Dir Acc: 30.0%


Segment 23/24 - RMSE: 175.38 | MAPE: 1.35% | R2: 0.2232 | Dir Acc: 43.3%


Segment 24/24 - RMSE: 115.14 | MAPE: 0.87% | R2: 0.0523 | Dir Acc: 56.7%

--- Median Metrics for Kalman-v2 Fine-Tuned TimesFM on SSMI (post-2018 test) ---
Median RMSE:          208.7455
Median MAPE:          1.9106%
Median Pearson R2:    0.0680
Directional Accuracy: 387/720 days (53.75%)
